# LLM Accuracy on The Gamma Type Providers

This notebook analyses results from running `score.js` across four configurations: two models (claude-haiku-4-5, claude-opus-4-7) × two prompt settings (no system prompt, default system prompt). Each row in the result CSVs corresponds to one navigation chain — a sequence of member-pick steps that builds a complete data query in The Gamma type provider system.

The five provider types tested are `olympics`, `worldbank`, `expenditure`, `shared`, and `drWho`, each with a different navigation structure and difficulty profile.

## Loading results

Result files are discovered automatically from the `results/` directory. The model name and prompt label are parsed from the filename pattern `{model}__{prompt}__{provider}__{timestamp}.csv`. A `model` and `prompt` column are added to each file's dataframe before concatenation.

In [ ]:
import glob
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

results_dir = 'results'

frames = []
for path in sorted(glob.glob(os.path.join(results_dir, '*.csv'))):
    fname = os.path.basename(path)
    m = re.match(r'^(.+?)__(.+?)__(.+?)__\d+\.csv$', fname)
    if not m:
        continue
    model, prompt, _ = m.group(1), m.group(2), m.group(3)
    df = pd.read_csv(path)
    df['model'] = model
    df['prompt'] = prompt
    frames.append(df)

data = pd.concat(frames, ignore_index=True)
print(f'{len(data)} chains from {len(frames)} file(s)')
data.head()

## Computing accuracy statistics

Chain-level accuracy is `correct_steps / total_steps`. Aggregation by `provider × model × prompt` sums the raw step counts (for the mean) and takes the standard deviation of per-chain accuracies (for error bars). An `overall` row sums across all providers.

Two providers — `drWho` and `expenditure` — have very few chains (often just one per snippet) so their standard deviations are unreliable. They are excluded from per-provider charts but remain in the `overall` aggregate.

Model/prompt configs are defined here with display labels and paired colors (light/dark blue for Haiku, light/dark orange for Opus) so all chart cells can reference them consistently.

In [ ]:
# Per-chain accuracy (unit of variance)
data['chain_acc'] = data['correct_steps'] / data['total_steps']

# Providers excluded from per-provider charts (too few chains)
EXCLUDE = {'drWho', 'expenditure'}

# Model/prompt configs: (model, prompt, label, color)
# Light/dark pairs: blue = haiku, orange = opus
ALL_CONFIGS = [
    ('claude-haiku-4-5',  'no-prompt',      'claude-haiku-4-5, no prompt',   '#91BFD9'),
    ('claude-haiku-4-5',  'default-prompt', 'claude-haiku-4-5, with prompt', '#1F6B9A'),
    ('claude-opus-4-7',   'no-prompt',      'claude-opus-4-7, no prompt',    '#F4A97A'),
    ('claude-opus-4-7',   'default-prompt', 'claude-opus-4-7, with prompt',  '#C45A0A'),
]
present = set(zip(data['model'], data['prompt']))
configs = [(m, p, lbl, clr) for m, p, lbl, clr in ALL_CONFIGS if (m, p) in present]

# Accuracy + std per provider x model x prompt
by_provider = (
    data
    .groupby(['provider', 'model', 'prompt'], as_index=False)
    .agg(total=('total_steps', 'sum'), correct=('correct_steps', 'sum'), std=('chain_acc', 'std'))
)
by_provider['accuracy'] = by_provider['correct'] / by_provider['total']

# Overall (all providers) per model x prompt
overall = (
    data
    .groupby(['model', 'prompt'], as_index=False)
    .agg(total=('total_steps', 'sum'), correct=('correct_steps', 'sum'), std=('chain_acc', 'std'))
)
overall['accuracy'] = overall['correct'] / overall['total']
overall.insert(0, 'provider', 'overall')

stats = pd.concat([by_provider, overall], ignore_index=True)
stats['std'] = stats['std'].fillna(0)  # single-chain groups have no meaningful spread
stats

## Accuracy by provider

Grouped bar chart showing the fraction of steps the LLM picks correctly, broken down by provider. Error bars show one standard deviation of per-chain accuracy within each group — wide bars indicate high variance across snippets. The dashed separator distinguishes per-provider results from the aggregate `overall` column.

In [ ]:
provider_order = sorted(p for p in data['provider'].unique() if p not in EXCLUDE) + ['overall']
n_providers = len(provider_order)
n_configs   = len(configs)
bar_w       = 0.18
x           = np.arange(n_providers)

fig, ax = plt.subplots(figsize=(10, 5))
for i, (model, prompt, label, color) in enumerate(configs):
    subset = stats[(stats['model'] == model) & (stats['prompt'] == prompt)]
    acc, err = [], []
    for prov in provider_order:
        row = subset[subset['provider'] == prov]
        acc.append(row['accuracy'].iloc[0] * 100 if len(row) else 0.0)
        err.append(row['std'].iloc[0]    * 100 if len(row) else 0.0)
    offset = (i - (n_configs - 1) / 2) * bar_w
    ax.bar(x + offset, acc, bar_w * 0.9, label=label, color=color, zorder=3)
    ax.errorbar(x + offset, acc, yerr=err, fmt='none', ecolor='#333',
                elinewidth=1.2, capsize=3, zorder=4)

ax.axvline(x=n_providers - 1 - 0.5, color='#aaa', linewidth=1, linestyle='--', zorder=2)
ax.set_xticks(x)
ax.set_xticklabels(provider_order, fontsize=11)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylim(0, 115)
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_title('LLM accuracy at navigating The Gamma type providers', fontsize=13)
ax.legend(fontsize=10)
ax.grid(axis='y', linestyle=':', alpha=0.6, zorder=0)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('results/accuracy.png', dpi=150)
plt.show()

## Prompt lift by provider

Difference in accuracy between the *with-prompt* and *no-prompt* runs, shown separately for each model. Positive values mean the system prompt helped; a negative bar would indicate it hurt. This makes it easy to see which providers benefit most from the per-provider navigation rules in the system prompt, and whether Opus needs the prompt less than Haiku.

In [ ]:
provider_order_lift = sorted(p for p in data['provider'].unique() if p not in EXCLUDE) + ['overall']

lift_rows = []
for prov in provider_order_lift:
    for model in data['model'].unique():
        sub = stats[(stats['provider'] == prov) & (stats['model'] == model)]
        r_with = sub[sub['prompt'] == 'default-prompt']
        r_no   = sub[sub['prompt'] == 'no-prompt']
        if len(r_with) and len(r_no):
            lift_rows.append({
                'provider': prov, 'model': model,
                'lift': (r_with['accuracy'].iloc[0] - r_no['accuracy'].iloc[0]) * 100,
            })
lift = pd.DataFrame(lift_rows)

# One bar per model; use the darker (with-prompt) color
model_style = {m: (lbl.replace(', with prompt', ''), clr)
               for m, p, lbl, clr in configs if p == 'default-prompt'}
seen, models_ordered = set(), []
for m, *_ in configs:
    if m not in seen and m in model_style:
        seen.add(m); models_ordered.append(m)

n_p   = len(provider_order_lift)
bar_w = 0.3
x     = np.arange(n_p)

fig, ax = plt.subplots(figsize=(10, 4))
for i, model in enumerate(models_ordered):
    label, color = model_style[model]
    lifts = [lift[(lift['provider'] == p) & (lift['model'] == model)]['lift'].iloc[0]
             if len(lift[(lift['provider'] == p) & (lift['model'] == model)]) else 0.0
             for p in provider_order_lift]
    offset = (i - (len(models_ordered) - 1) / 2) * bar_w
    ax.bar(x + offset, lifts, bar_w * 0.9, label=label, color=color, zorder=3)

ax.axhline(0, color='#666', linewidth=0.8, zorder=2)
ax.axvline(x=n_p - 1 - 0.5, color='#aaa', linewidth=1, linestyle='--', zorder=2)
ax.set_xticks(x)
ax.set_xticklabels(provider_order_lift, fontsize=11)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylabel('Accuracy lift (%)', fontsize=11)
ax.set_title('Accuracy gain from adding the system prompt', fontsize=13)
ax.legend(fontsize=10)
ax.grid(axis='y', linestyle=':', alpha=0.6, zorder=0)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('results/prompt_lift.png', dpi=150)
plt.show()

## Accuracy vs chain length

Each point is one chain. The x-axis is the number of steps in that chain; the y-axis is the fraction answered correctly. A downward trend would indicate that longer chains are harder to navigate end-to-end. Clustering at 0 % and 100 % would suggest the task behaves in an all-or-nothing fashion for many chains.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for model, prompt, label, color in configs:
    sub = data[(data['model'] == model) & (data['prompt'] == prompt)]
    ax.scatter(sub['total_steps'], sub['chain_acc'] * 100,
               alpha=0.45, color=color, label=label, s=25, zorder=3)

ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_xlabel('Chain length (steps)', fontsize=11)
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_title('Accuracy vs chain length', fontsize=13)
ax.legend(fontsize=10)
ax.grid(linestyle=':', alpha=0.6, zorder=0)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('results/accuracy_vs_length.png', dpi=150)
plt.show()

## Hardest snippets

The 15 snippets with the lowest average accuracy, aggregated across all four model/prompt configurations. These are the best candidates for closer inspection: are they inherently ambiguous navigation paths, or could better hints or prompt wording resolve them?

In [ ]:
snippet_stats = (
    data
    .groupby(['snippet_id', 'snippet_title'], as_index=False)
    .agg(total=('total_steps', 'sum'), correct=('correct_steps', 'sum'))
)
snippet_stats['accuracy'] = snippet_stats['correct'] / snippet_stats['total']
hardest = snippet_stats.sort_values('accuracy').head(15).reset_index(drop=True)
hardest['accuracy'] = hardest['accuracy'].map('{:.1%}'.format)
hardest.rename(columns={
    'snippet_id':    'ID',
    'snippet_title': 'Title',
    'total':         'Total steps',
    'correct':       'Correct steps',
    'accuracy':      'Accuracy',
}, inplace=True)
hardest

## Per-chain accuracy: Haiku vs Opus

Each point is one chain scored with the default system prompt. Points above the diagonal are chains where Opus outperforms Haiku; points below are the reverse. Clusters near (0 %, 0 %) identify chains that are hard for both models — likely structurally ambiguous navigation paths rather than a model capability gap. Color encodes the provider.

In [ ]:
haiku_acc = (
    data[(data['model'] == 'claude-haiku-4-5') & (data['prompt'] == 'default-prompt')]
    [['snippet_id', 'chain_index', 'provider', 'chain_acc']]
    .rename(columns={'chain_acc': 'haiku_acc'})
)
opus_acc = (
    data[(data['model'] == 'claude-opus-4-7') & (data['prompt'] == 'default-prompt')]
    [['snippet_id', 'chain_index', 'provider', 'chain_acc']]
    .rename(columns={'chain_acc': 'opus_acc'})
)
corr = haiku_acc.merge(opus_acc, on=['snippet_id', 'chain_index', 'provider'])

if corr.empty:
    print('No overlapping chains between the two models yet — re-run once both are complete.')
else:
    provider_colors = {
        'olympics':    '#E15759',
        'worldbank':   '#4E79A7',
        'shared':      '#59A14F',
        'drWho':       '#B07AA1',
        'expenditure': '#F28E2B',
    }
    fig, ax = plt.subplots(figsize=(6, 6))
    for prov, group in corr.groupby('provider'):
        ax.scatter(group['haiku_acc'] * 100, group['opus_acc'] * 100,
                   alpha=0.6, label=prov, color=provider_colors.get(prov, '#aaa'), s=35, zorder=3)
    ax.plot([0, 100], [0, 100], color='#aaa', linewidth=1, linestyle='--', zorder=2)
    ax.set_xlabel('claude-haiku-4-5 accuracy (%)', fontsize=11)
    ax.set_ylabel('claude-opus-4-7 accuracy (%)', fontsize=11)
    ax.set_title('Per-chain accuracy: Haiku vs Opus (with prompt)', fontsize=12)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.legend(fontsize=10)
    ax.grid(linestyle=':', alpha=0.6, zorder=0)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.savefig('results/haiku_vs_opus.png', dpi=150)
    plt.show()